[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session6/student/Lab2_RAG_Evaluation_DeepEval_Student.ipynb)


# Lab 2: RAG Evaluation with DeepEval
## CCI Session 6

**Duration:** 15 minutes

### Clinical Scenario
> Your RAG system from Lab 1 looks like it works — but does it really? In clinical AI, confident wrong answers harm patients. Today you measure 4 critical RAG quality metrics on the same Wilms tumor RAG system.

### Objective
- Build test set of ground-truth clinical Q&A pairs
- Apply DeepEval's 4 core metrics
- Compare a 'bad' RAG vs 'good' RAG
- Identify common failure modes

---
## Setup

Install packages, add **`OPENAI_API_KEY`** to Colab Secrets. Files are loaded automatically from **Google Drive** (`MyDrive/CCI_Session6/`). Manual upload fallback is commented out in each cell if Drive is unavailable.

In [ ]:
!pip install -q llama-parse langchain langchain-text-splitters langchain-openai langchain-community langchain-chroma chromadb deepeval

In [ ]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
# os.environ['LLAMA_CLOUD_API_KEY'] = userdata.get('LLAMA_CLOUD_API_KEY')  # only needed if re-parsing

In [ ]:
from langchain_core.documents import Document

# ── Option 1: Load from Google Drive (no upload needed) ──────────────────────────────────────────
from google.colab import drive
import json
drive.mount('/content/drive')
with open('/content/drive/MyDrive/CCI_Session6/wt_parsed.json', encoding='utf-8') as f:
    lc_docs = [Document(page_content=d['text'], metadata=d['metadata']) for d in json.load(f)]
print(f"Loaded {len(lc_docs)} docs from Google Drive")
# ─────────────────────────────────────────────────────────────────────────────────────────────────

# ── Option 2: Upload WT_parsed.md from your PC ───────────────────────────────────────────────────
# from google.colab import files
# files.upload()  # pick WT_parsed.md
# with open('WT_parsed.md', encoding='utf-8') as f:
#     pages = f.read().split('\n\n---\n\n')
# lc_docs = [Document(page_content=p, metadata={'source': 'WT.pdf'}) for p in pages if p.strip()]
# print(f"Loaded {len(lc_docs)} docs from WT_parsed.md")
# ─────────────────────────────────────────────────────────────────────────────────────────────────

# ── Option 3: Upload wt_parsed.json from your PC ─────────────────────────────────────────────────
# from google.colab import files
# import json
# files.upload()  # pick wt_parsed.json
# with open('wt_parsed.json', encoding='utf-8') as f:
#     lc_docs = [Document(page_content=d['text'], metadata=d['metadata']) for d in json.load(f)]
# ─────────────────────────────────────────────────────────────────────────────────────────────────

---
## Cell 1 — Ground-truth test set

Define several **Wilms tumor** Q&A pairs. Each row: `input`, `expected_output`, and `retrieval_context` (short gold strings). These drive DeepEval and make regressions visible when you change chunking or `k`.


In [ ]:
# TODO: define 5 clinical Q&A pairs about Wilms tumor with expected answers.
# Each entry: {'input', 'expected_output', 'retrieval_context' (list[str] of ground-truth excerpts)}
test_cases = [
    {
        'input': 'What chemotherapy regimen is recommended for Stage IV Wilms tumor?',
        'expected_output': '...',
        'retrieval_context': ['...']
    },
    # TODO: add 4 more (Stage I FH, vincristine toxicity, anaplastic histology treatment, radiation indication)
]
print(f'{len(test_cases)} test cases defined')

---
## Shared docs (loaded in Setup)

`lc_docs` is already loaded from `wt_parsed.json`. Both pipelines re-chunk it differently — only **chunking** changes between BAD and GOOD.

In [ ]:
# lc_docs already loaded from wt_parsed.json (see Setup cell above)
print(f"lc_docs ready: {len(lc_docs)} docs")

# ── optional: re-parse from scratch (uncomment + set PDF_PATH if wt_parsed.json unavailable) ─────
# from llama_parse import LlamaParse
# from langchain_core.documents import Document
# parser = LlamaParse(api_key=os.environ['LLAMA_CLOUD_API_KEY'], result_type='markdown')
# raw_docs = parser.load_data(PDF_PATH)
# lc_docs = [Document(page_content=d.text, metadata={'source': 'WT.pdf'}) for d in raw_docs]
# ─────────────────────────────────────────────────────────────────────────────────────────────────

---
## Cell 2 — BAD RAG (chunk_size=100, no overlap)

On purpose, chunks are **too small**: facts and dosing context get split across many segments, so retrieval often returns **incomplete** context. Expect weaker DeepEval scores.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Answer ONLY from context.\n\n{context}'),
    ('human', '{input}')
])

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

def make_rag_chain(retriever, prompt, llm):
    _answer = prompt | llm | StrOutputParser()
    def _fn(d):
        docs = retriever.invoke(d['input'])
        return {'answer': _answer.invoke({'context': format_docs(docs), 'input': d['input']}), 'context': docs}
    return RunnableLambda(_fn)

# TODO: build bad pipeline with chunk_size=100, chunk_overlap=0
# Hint: bad_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0)
bad_splitter = ...
bad_chunks = ...
bad_vs = Chroma.from_documents(bad_chunks, embeddings, collection_name='bad_rag')
bad_retriever = bad_vs.as_retriever(search_kwargs={'k': 4})
bad_chain = make_rag_chain(bad_retriever, prompt, llm)
print(f'Bad RAG: {len(bad_chunks)} tiny chunks')

---
## Cell 3 — GOOD RAG (chunk_size=1000, overlap 200)

Match Lab 1 defaults so each chunk usually holds a **coherent** paragraph or table fragment. Keep `source` metadata on each `Document` for traceability.


In [ ]:
from google.colab import drive
import zipfile

drive.mount('/content/drive')  # already mounted above — safe to call again
DRIVE_ZIP = '/content/drive/MyDrive/CCI_Session6/chroma_wt.zip'
with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
    z.extractall('.')
print("chroma_wt extracted → ./chroma_wt")

# ── alternative: manual upload (if Drive not available) ──────────────────────────────────────────
# from google.colab import files
# files.upload()  # pick chroma_wt.zip (from Lab 1)
# with zipfile.ZipFile('chroma_wt.zip', 'r') as z:
#     z.extractall('.')
# ─────────────────────────────────────────────────────────────────────────────────────────────────

In [ ]:
# Good RAG — load from Lab 1's persisted vector store (chunk_size=1000, chunk_overlap=200)
good_vs = Chroma(
    collection_name='wt_good',
    embedding_function=embeddings,
    persist_directory='./chroma_wt',
)
good_retriever = good_vs.as_retriever(search_kwargs={'k': 4})
good_chain = make_rag_chain(good_retriever, prompt, llm)
print(f'Good RAG: {good_vs._collection.count()} chunks (loaded from Lab 1)')

# ── optional: rebuild from scratch (uncomment if ./chroma_wt doesn't exist) ──────────────────────
# good_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# good_chunks = good_splitter.split_documents(lc_docs)
# good_vs = Chroma.from_documents(good_chunks, embeddings, collection_name='wt_good', persist_directory='./chroma_wt')
# ─────────────────────────────────────────────────────────────────────────────────────────────────

---
## Cell 4 — DeepEval metrics

Instantiate **4 core metrics** with `threshold=0.7` and `model='gpt-4o-mini'` as the judge.

| Metric | What it measures |
|--------|-----------------|
| `FaithfulnessMetric` | Is the answer grounded in the retrieved context? (low → hallucination) |
| `AnswerRelevancyMetric` | Does the answer actually address the question? (low → off-topic) |
| `ContextualRelevancyMetric` | Are the retrieved chunks about the question? (low → wrong retrieval) |
| `ContextualRecallMetric` | Do the chunks cover what the reference answer requires? (low → missing facts) |

**Hint:** `FaithfulnessMetric(threshold=0.7, model='gpt-4o-mini')`

In [ ]:
from deepeval.metrics import (
    FaithfulnessMetric, AnswerRelevancyMetric,
    ContextualRelevancyMetric, ContextualRecallMetric,
)
from deepeval.test_case import LLMTestCase

# TODO: instantiate the 4 metrics with threshold=0.7, model='gpt-4o-mini'
faithfulness = ...
answer_relevancy = ...
ctx_relevancy = ...
ctx_recall = ...
metrics = [faithfulness, answer_relevancy, ctx_relevancy, ctx_recall]

---
## Cell 5 — Run evaluations on both pipelines

Loop over `test_cases`, run each through both chains, and score with all 4 metrics. The `run_pipeline` helper is already written — use it to fill the `actual, ret_ctx = ...` line.

**Hint:** `actual, ret_ctx = run_pipeline(chain, tc)` then pass them into `LLMTestCase`.

In [ ]:
def run_pipeline(chain, tc):
    res = chain.invoke({'input': tc['input']})
    return res['answer'], [d.page_content for d in res['context']]

def evaluate(chain, label):
    rows = []
    for tc in test_cases:
        # TODO: run pipeline → get actual_output and retrieval_context
        actual, ret_ctx = ...
        # TODO: build LLMTestCase
        case = LLMTestCase(
            input=tc['input'],
            actual_output=actual,
            expected_output=tc['expected_output'],
            retrieval_context=ret_ctx,
        )
        scores = {}
        for m in metrics:
            m.measure(case)
            scores[m.__class__.__name__] = m.score
        scores['question'] = tc['input'][:60]
        rows.append(scores)
    return rows

bad_results = evaluate(bad_chain, 'BAD')
good_results = evaluate(good_chain, 'GOOD')

---
## Cell 6 — Compare results

Print mean scores for BAD vs GOOD. Look for the biggest gap — it's usually **ContextualRecall** because tiny chunks (BAD) miss the full fact needed to match the reference answer.

**What does success look like?** GOOD scores consistently ≥ 0.7 across all 4 metrics, while BAD scores drop noticeably, especially on recall.

In [ ]:
import pandas as pd
df_bad = pd.DataFrame(bad_results)
df_good = pd.DataFrame(good_results)
print('BAD RAG mean scores:')
print(df_bad.drop(columns=['question']).mean())
print('\nGOOD RAG mean scores:')
print(df_good.drop(columns=['question']).mean())

---
## Cell 7 — Diagnose failures

Find the worst BAD case and print its retrieved chunks. Ask yourself:
- Are the chunks too short to contain the full clinical fact?
- Would the LLM hallucinate to fill the gap?

**Hint:** `worst_idx = df_bad['FaithfulnessMetric'].idxmin()` then print `test_cases[worst_idx]` and the retrieved context.

In [ ]:
# TODO: for the lowest-scoring case in BAD RAG, print question + retrieved chunks
# Discuss: why did the metric fail? small chunks → loss of context
...

## Stretch — Add a custom G-Eval metric (e.g., 'clinical safety')

## KHCC Connection
Confident wrong answers harm patients. DeepEval makes RAG quality measurable so the AI Office can sign off on a clinical assistant before it sees real patients.